Build a Baseline CNN Model
---

## Overview
In this task we build a **Convolutional Neural Network (CNN) from scratch** as our baseline model.
A baseline gives us a performance floor to compare against when we apply more advanced techniques like Transfer Learning in Task 5.

## What is a CNN?
A CNN is a type of deep learning model designed specifically for image data.
Unlike regular neural networks that see a flat list of numbers, a CNN understands
the **spatial structure** of an image — it learns that nearby pixels are related.

It does this through three key operations:
- **Conv2D** — slides small filters across the image to detect patterns like edges and textures
- **MaxPooling2D** — shrinks the image by keeping only the strongest features
- **Dense** — a fully connected layer that uses learned features to make the final classification

## Architecture
```
Input (224x224x3)
    ↓
Block 1: Conv2D(32) → BatchNorm → MaxPool  → (112x112x32)
    ↓
Block 2: Conv2D(64) → BatchNorm → MaxPool  → (56x56x64)
    ↓
Block 3: Conv2D(128) → BatchNorm → MaxPool → (28x28x128)
    ↓
Flatten → Dense(256) → Dropout(0.5) → Dense(1, sigmoid)
    ↓
Output: probability between 0 (NORMAL) and 1 (PNEUMONIA)
```


---
## Step 1: Import Libraries and Setup
**What:** Import all required modules from our `src/` pipeline.

**Why:** We use our modular `src/` files instead of rewriting code — this keeps the notebook clean and the logic reusable across tasks.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/pneumonia-detection')
from src.data_loader import build_generators
from src.model_builder import build_baseline_cnn
from src.trainer import train_model
from src.evaluator import plot_history
from src.config import BASELINE_MODEL_PATH, EPOCHS
print('All modules loaded successfully.')

---
## Step 2: Load Data Generators
**What:** Build train, validation and test generators using our preprocessing pipeline from Task 2.

**Why:** We need to feed data into the model in batches. The generators handle resizing, normalisation and augmentation automatically.


In [ ]:
train_gen, val_gen, test_gen = build_generators()
print(f'Training batches  : {len(train_gen)}')
print(f'Validation batches: {len(val_gen)}')
print(f'Test batches      : {len(test_gen)}')
print(f'Class indices     : {train_gen.class_indices}')

---
## Step 3: Build the Baseline CNN
**What:** Construct a 3-block CNN using Keras Sequential API.

**Why:** We build from scratch first to establish a baseline. Each block increases the number of filters (32→64→128) because deeper layers need to detect more complex and abstract features.

Key design choices:
- `BatchNormalization` — stabilises training by normalising activations at each layer
- `Dropout(0.5)` — randomly disables 50% of neurons during training to prevent overfitting
- `sigmoid` output — produces a probability between 0 and 1 for binary classification
- `BinaryCrossentropy` loss — standard loss function for 2-class problems


In [ ]:
baseline_model = build_baseline_cnn()
baseline_model.summary()

---
## Step 4: Train the Model
**What:** Fit the model on training data, monitor performance on validation data.

**Why:** Training is where the model learns — it adjusts its weights through backpropagation to minimise the loss function.

Two callbacks are used:
- **EarlyStopping** — stops training when `val_loss` stops improving (patience=3). Prevents wasting time and overfitting.
- **ReduceLROnPlateau** — halves the learning rate when the model gets stuck. Helps escape plateaus.

> Note: Training will take a few minutes depending on your hardware.


In [ ]:
history = train_model(
    baseline_model, train_gen, val_gen,
    epochs=EPOCHS, label='Baseline CNN'
)

---
## Step 5: Plot Training History
**What:** Visualise accuracy and loss curves for both training and validation sets.

**Why:** The training curves reveal critical information:
- If train accuracy >> val accuracy → **overfitting** (model memorised training data)
- If both curves improve together → **good generalisation**
- The point where val loss stops improving and diverges is where EarlyStopping triggers


In [ ]:
plot_history(history, title='Baseline CNN - Training History')

---
## Step 6: Save the Model
**What:** Save the trained model weights to disk.

**Why:** Saving the model means we can reload it in Task 4 for evaluation without retraining.
We save to `outputs/baseline_cnn.h5` as defined in `src/config.py`.


In [ ]:
baseline_model.save(BASELINE_MODEL_PATH)
print(f'Model saved to: {BASELINE_MODEL_PATH}')

---
## Observations

**1. What does each block of Conv2D → BatchNorm → MaxPooling do?**
- **Conv2D**: Applies learnable filters to extract features like edges, textures and patterns. Each filter produces an activation map highlighting where a specific feature is detected.
- **BatchNormalization**: Normalizes activations at each mini-batch, stabilizing and accelerating training by reducing internal covariate shift. Allows higher learning rates.
- **MaxPooling2D**: Downsamples by taking the maximum value over a 2x2 window, reducing spatial dimensions and computational cost while retaining dominant features.

Together: Conv2D extracts features, BatchNorm stabilizes learning, MaxPooling reduces dimensionality.

**2. Total parameters — what does that number mean?**
Total parameters are all weights and biases the network learns during training. They define model capacity and complexity. More parameters means a more complex model capable of learning intricate patterns, but with increased overfitting risk on small datasets.

**3. What did the training curves tell you about overfitting?**
The curves show clear overfitting:
- Training Accuracy: consistently high (~93-95%) and keeps improving
- Validation Accuracy: fluctuates and stays significantly lower (~81-87%)
- Training Loss: steadily decreases
- Validation Loss: initially decreases then diverges upward

The gap between train and val metrics is a strong sign the model memorizes training data rather than generalizing to unseen images.

**4. Why did EarlyStopping trigger at epoch 10?**
EarlyStopping was configured with `patience=3` monitoring `val_loss`:
- Epoch 7: val_loss 0.2396 — best performance
- Epoch 8: val_loss 0.2644 — no improvement
- Epoch 9: val_loss 0.4211 — no improvement
- Epoch 10: val_loss 0.3564 — no improvement

After 3 consecutive epochs without improvement, training stopped and `restore_best_weights=True` automatically loaded epoch 7 weights — the best version of the model.

---
> **Next Step:** Task 4 evaluates this model on the test set using clinical metrics like recall, precision and F1-score.
